# 9. Exportacion a Power BI

Proposito: consolidar los marts de `data/gold/` en archivos parquet **unicos**
optimizados para el conector nativo de Power BI, en `data/powerbi/` (+ `manifest.json`).

Equivale a `python main.py powerbi --drop`. Requiere haber corrido antes **gold**.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: c:\Users\user\Downloads\EP-GDM-G6
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot


In [2]:
from app.pipeline.powerbi import PowerBIPipeline
from app.utils.spark import SparkClient

# Equivale a:  python main.py powerbi --drop
# Lee los marts de data/gold/ y los consolida en archivos parquet UNICOS
# (coalesce(1) + renombrado del part-file) en data/powerbi/, mas un manifest.json.
# Requiere haber corrido antes gold (06_run_gold.ipynb o `python main.py gold`).
spark_client = SparkClient()
PowerBIPipeline(spark_client).run(drop=True)
spark = spark_client.get_session()
print("PowerBI export completado -> data/powerbi/")

2026-06-19 16:04:22,335 - INFO - Iniciando PowerBI Pipeline
2026-06-19 16:04:22,337 - INFO - Exportando DIM_CALENDARIO ...
2026-06-19 16:04:26,900 - INFO -   DIM_CALENDARIO: 312 filas — 4.6 KB
2026-06-19 16:04:26,901 - INFO - Exportando DIM_ANIO ...
2026-06-19 16:04:27,325 - INFO -   DIM_ANIO: 18 filas — 0.5 KB
2026-06-19 16:04:27,331 - INFO - Exportando DIM_GEOGRAFIA ...
2026-06-19 16:04:27,851 - INFO -   DIM_GEOGRAFIA: 1,892 filas — 36.5 KB
2026-06-19 16:04:27,851 - INFO - Exportando MART_INGRESOS_GEOGRAFICO ...
2026-06-19 16:04:29,486 - INFO -   MART_INGRESOS_GEOGRAFICO: 122,475 filas — 3870.9 KB
2026-06-19 16:04:29,486 - INFO - Exportando MART_INGRESOS_CLASIFICADOR ...
2026-06-19 16:04:29,918 - INFO -   MART_INGRESOS_CLASIFICADOR: 2,244 filas — 45.9 KB
2026-06-19 16:04:29,919 - INFO - Exportando MART_INGRESOS_EJECUTORA ...
2026-06-19 16:04:30,353 - INFO -   MART_INGRESOS_EJECUTORA: 11,346 filas — 443.7 KB
2026-06-19 16:04:30,353 - INFO - Exportando MART_PREDIAL ...
2026-06-19 16:04

PowerBI export completado -> data/powerbi/


In [4]:
import json
from pathlib import Path
import pyarrow.parquet as pq

powerbi_dir = Path("data/powerbi")
manifest = json.loads((powerbi_dir / "manifest.json").read_text(encoding="utf-8"))
print(f"Exportado : {manifest['exported_at']}")
print(f"Tablas    : {len(manifest['tables'])}")

total_rows = 0
for f in sorted(powerbi_dir.glob("*.parquet")):
    md = pq.read_metadata(f)
    total_rows += md.num_rows
    print(f"{f.name:<32} {md.num_rows:>12,} filas   {f.stat().st_size/1024:>10.1f} KB")
print(f"Total     : {total_rows:,} filas")

Exportado : 2026-06-19T21:04:22.337754+00:00
Tablas    : 8
DIM_ANIO.parquet                           18 filas          0.5 KB
DIM_CALENDARIO.parquet                    312 filas          4.6 KB
DIM_GEOGRAFIA.parquet                   1,892 filas         36.5 KB
MART_INGRESOS_CLASIFICADOR.parquet        2,244 filas         45.9 KB
MART_INGRESOS_EJECUTORA.parquet        11,346 filas        443.7 KB
MART_INGRESOS_GEOGRAFICO.parquet      122,475 filas       3870.9 KB
MART_PREDIAL.parquet                  250,521 filas       1864.8 KB
MART_RENAMU.parquet                12,770,291 filas      80257.5 KB
Total     : 13,159,099 filas
